## Model Problem A: Gaussian Targets

Assuming that 
$$
    \mu_0 = N(0, C) \quad \mu = N(m, \tilde{C})
$$
and then we can write
$$
    \mu(dx) \propto \exp( - \Phi(x)) \mu_0(dx)
$$
where 
$$
    \Phi(x) = \frac{1}{2} \langle (\tilde{C}^{-1} + C^{-1})x, x \rangle  
    - \langle \tilde{C}^{-1} x , m \rangle.
$$

## Model Problem B: Find the Surface

We consider the statistical inversion problem of estimating $\mathbf{x} \in \mathbb{R}^n$ from data $z \in \mathbb{R}$ gathered according the measurement model:
$$
    z = f(\mathbf{x}) + \eta     \tag{MM1}
$$    
where
$$    
    \eta \sim N(0, \sigma^2) \text{ and } f: \mathbb{R}^n \to \mathbb{R}
    \text{ is the forward function. }
$$
Leaving aside the measurement noise $\eta$ we would like to recover the preimage $f^{-1}(z)$.  We constrain this set with Gaussian prior $\mu_0 \sim N(0, C)$ on $\mathbf{x}$. Then the posterior 'bayesian solution' $\mathbf{x}|z$ is the distribution:
$$
    \mu(d\mathbf{x}) \propto \exp\left( - \frac{1}{2 \sigma^2} ( f(\mathbf{x}) - z)^2 
    - \frac{1}{2} <C^{-1}\mathbf{x}, \mathbf{x}>\right) d\mathbf{x}.
    \tag{S1}
$$

## Model Problem C: Matrix Coefficents from Solutions

We next consider inverse problem of estimating the coefficents of an $n \times n$ anti-semetric matrix $A$ from data $\theta \in \mathbb{R}^n$ gathered according the measurement model:
$$
    z = \mathcal{O}(\theta(A)) + \eta \qquad \text{ where } \eta \sim N(0, \sigma^2)
    \tag{MM2}
$$
Here $\theta := \theta(A)$ is the solution of
$$
    (A + \kappa I)\theta = g
$$
and where $\mathcal{O}(\theta)$ defined by $\mathcal{O}: \mathbb{R}^n \to \mathbb{R}^m$ is a partial observation of $\theta$. Typically
$$
    \mathcal{O}((\theta_1, \ldots, \theta_n)) = (\theta_1,\ldots, \theta_m)
$$
Here the Bayesian posterior will take the form
$$
    \mu(dA) \propto \exp\left( - \frac{1}{2 \sigma^2} 
    | \mathcal{O}( (A +\kappa I)^{-1} g) - z|^2 
    - \frac{1}{2} <C^{-1}A, A>\right) dA.
    \tag{S1}
$$
which is a target in $d = n(n-1)/2$






To Do:

*Add experiments of type A for the mean shifts
*Add experiments for AD with stranger observations
*Add high dimensional AD experiment.

In [1]:
%load_ext autoreload
%autoreload 2

#Set-up: Packages to Import and Untility Functions

#Core Numerical Packages for Paral
import MCMC_Sampliers_Testing as MCMCsmp
import Utilities as Utils

#Numerical Elements
from numpy.linalg import norm
import numpy as np
from numpy import dot, array, transpose, diag
import random
import math

#Input Output utils
import os
import pandas as pd

#Stats elements
#from scipy.stats import norm
from scipy.stats import chi2
rng = np.random.default_rng()

#Plotting stuff
from matplotlib.patches import Ellipse

#Finished Message
import smtplib
from email.message import EmailMessage

from functools import partial
import multiprocessing as mp


#Save Locations
#MacbookPro
#saveLocBase = "/Users/negh_macpro/Library/CloudStorage/Dropbox/MCMC_Runs/" 
#saveLocBaseData = "/Users/negh_macpro/Library/CloudStorage/Dropbox/MCMC_Runs/Data/"

#Ubuntu Work Machine
saveLocBase = "/home/neghadmin/Dropbox/MCMC_Runs/"
saveLocBaseData = "/home/neghadmin/Dropbox/MCMC_Runs/Data/"


def run_p_Sweep_Batches(ImpLists, q0Fn, TarDim, CovPrior, Pot, saveFileBase,saveFileBaseData,tsComp,burnIn):
    for ImpList in ImpLists:
        curPList = [I[0] for I in ImpList]
        print("Running Blocks for: " + str(curPList))
        saveDictLocCur = saveFileBaseData + "data_dict_info_p_"
        for p in curPList:
           saveDictLocCur += str(p) + "_" 
        saveDictLocCur += ".npy"
        if __name__ == "__main__":
            par_sweep_dict = Utils.parameter_sweep_p_rho(ImpList, q0Fn, TarDim, CovPrior, Pot, saveDictExt=True,
                                                           saveDictLoc = saveDictLocCur)

        #par_sweep_dictEx1 = np.load(saveDictLocCur, allow_pickle=True).item()

        Utils.parameter_sweep_p_rho_save_figures(par_sweep_dict, TarDim, tsComp,burnIn, saveFileBase)


#FORMAT: [p,NumRho,NumSampsESS,numChainsESS, NumChainsgM]
#            -p: value to p to run study
#            -NumRho: 1/NumRho specfies the step size in rho over [0,1] for the study
#            -NumSampsESS: Length of the MCMC at each rho value to compute ESS/MSJD
#            -numChainsESS: Number of independent chains to compute ESS/MSJD
#            -NumChainsgM: Number of separate chain M= NumChainsgM to compute Var(\bar{g}_N^\rho)

#Test Run
#ImpList1 = [[3,10,5000,5,5], [7,10,5000,5,5]]
#ImpList2 = [[9,10,5000,5,5], [12,10,5000,5,5]]
#ImpLists = [ImpList1, ImpList2]

#Hardcore Run
ImpList_1 = [[7,10,2000,5,20]]
ImpList0 = [[5,50,50000,100,5000], [10,50,50000,100,5000]]
ImpList1 = [[20,50,50000,100,5000], [40,50,50000,100,5000]]
ImpList2 = [[80,50,10000,10,5000], [160,50,10000,10,5000]]
ImpList3 = [[320,50,10000,10,5000], [640,50,10000,10,5000]]
ImpList4 = [[1280,50,10000,10,5000]]
#ImpList5 = [[10000,25,5000,5,1000]]
ImpList5 = [[2560,25,5000,5,1000]]
ImpLists = [ImpList_1,ImpList0,ImpList1, ImpList2, ImpList3,ImpList4,ImpList5]
#ImpList5 = [[5000,50,50000,100,5000]]
#ImpList6 = [[10000,50,50000,100,5000]]
#ImpLists = [ImpList5,ImpList6]


def make_stat_measure(TarDim, priorCov,Pot, chainLn =20500, numChain =50,  MC_meth =MCMCsmp.MpCN, warmrho = .2, warmp = 500,burnIn=500,thinfactor=5):
    q0gen = lambda: np.random.multivariate_normal(np.zeros(TarDim), priorCov)
    MC_arg = [TarDim,priorCov,warmrho,Pot,warmp]
    samps, ESS = Utils.parallel_MCMC_Runs_ESS(chainLn,numChain,MC_meth,MC_arg,q0gen, burnIn)

    ESS_N_min = ESS.min()/(chainLn*numChain)
    print("Effective ESS//N: " + str(ESS_N_min))
    print()
    thin_smp = thinfactor*int(1/ESS_N_min)
    print("Number of Parallel Chains: " + str(numChain*thin_smp))
    print("Length per Chain: " + str(chainLn))
    print("Total samples to be generated: " + str(numChain*thin_smp*chainLn))
    thinned_samps = samps[::thin_smp]
    q0genFull = lambda:random.choice(thinned_samps)

    #parallel_MCMC_Runs(chainLn,numChain,MCMCmeth,MCMCmethArgs, q0gen, burn_In, thin)
    return Utils.parallel_MCMC_Runs(chainLn,numChain*thin_smp,MC_meth,MC_arg,q0genFull, burnIn,thinfactor)


print("Current cpu count:" + str(mp.cpu_count()))

Current cpu count:24


In [ ]:
#Set up Workflow for Type A experiments

def runAtypeExperiment(expId, modDim, priorCov, postMean, postCov, Pot, PlotComps,burnIn=1000):
    #Make Problem Information File
    ExpDir = "Large_p_study/Experiment_A/" + "Ex_ID_"+ str(expId) + "/"
    FileNmBase = saveLocBase+ExpDir
    FileNmBaseData =saveLocBaseData+ExpDir

    os.makedirs(FileNmBase, exist_ok=True)
    os.makedirs(FileNmBaseData, exist_ok=True)

    #Make Problem information String
    infoStr  = "Target Type:  Model Problem A -- Gaussian Target \n"
    infoStr += "Target Dimension: " + str(modDim) + "\n"
    infoStr += "Prior Covariance: \n" + str(priorCov) + "\n"
    infoStr += "Posterior Mean: " + str(postMean) + "\n"
    infoStr += "Posterior Covariance: \n" + str(postCov)+ "\n"

    Utils.message_ntfy("RUNNING EXAMPLE A" + str(expId) + "\n" + infoStr)

    #Save Problem Information
    probDataFile = FileNmBase + "Problem_Info.txt"
    with open(probDataFile, 'w') as file:
        file.write(infoStr)
        
    q0_stat_draw = lambda nmICs: np.random.multivariate_normal(postMean, postCov, size=nmICs)
    run_p_Sweep_Batches(ImpLists, q0_stat_draw, modDim, priorCov, Pot, FileNmBase,FileNmBaseData,PlotComps,burnIn)

In [ ]:
#Experiment A1
expIdA1 =  1
#Gaussian target-- Shifted Covariance on `low modes'

#Model dim
TarDimExA1 = 5
#Covariance algebraic decay
cov0 = 4
gam = 2
CovDiagPriorA1 = [cov0* (j**(-gam)) for j in list(range(1,TarDimExA1+1))]
CovPriorExA1 = Utils.mkDiagCov(CovDiagPriorA1)

#Information for Gaussian Posterior
#Perturb dim
pertDimExA1 = 2
CovDiagPostA1  = [CovDiagPriorA1[1], CovDiagPriorA1[0]] + CovDiagPriorA1[pertDimExA1:]
CovPostExA1 = Utils.mkDiagCov(CovDiagPostA1)
meanPostExA1 = np.zeros(TarDimExA1)

Pres_DiffA1 = np.diag(1/np.array(CovDiagPostA1) - 1/np.array(CovDiagPriorA1))
PotExA1Pass = partial(Utils.PotGaussPertCov, Pres_Diff =Pres_DiffA1 ,mode = "soft")

#runAtypeExperiment(expId, modDim, priorCov, postMean, postCov, PlotComps,burnIn=1000)

runAtypeExperiment(expIdA1,TarDimExA1, CovPriorExA1,meanPostExA1,CovPostExA1,PotExA1Pass,[0,1,4])


In [ ]:
#Experiment A2
expIdA2 = 2 
#Gaussian target-- Shifted Covariance on `low modes'--Higher Dimension

#Model dim
TarDimExA2 = 20
#Covariance algebraic decay
cov0 = 4
gam = 2
CovDiagPriorA2 = [cov0* (j**(-gam)) for j in list(range(1,TarDimExA2+1))]
CovPriorExA2 = Utils.mkDiagCov(CovDiagPriorA2)

#Information for Gaussian Posterior
#Perturb dim
pertDimExA2 = 2
CovDiagPostA2  = [CovDiagPriorA2[1], CovDiagPriorA2[0]] + CovDiagPriorA2[pertDimExA2:]
CovPostExA2 = Utils.mkDiagCov(CovDiagPostA2)
meanPostExA2 = np.zeros(TarDimExA2)

Pres_DiffA2 = np.diag(1/np.array(CovDiagPostA2) - 1/np.array(CovDiagPriorA2))
#print(Pres_DiffA2)
PotExA2Pass = partial(Utils.PotGaussPertCov, Pres_Diff =Pres_DiffA2 ,mode = "soft")

#runAtypeExperiment(expId, modDim, priorCov, postMean, postCov, PlotComps,burnIn=1000)

runAtypeExperiment(expIdA2,TarDimExA2, CovPriorExA2,meanPostExA2,CovPostExA2,PotExA2Pass,[0,1,4])


In [ ]:
#Experiment A3
expIdA3 = 3 
#Gaussian target-- Shifted Covariance on `low modes'--Higher Dimension

#Model dim
TarDimExA3 = 10
#Covariance algebraic decay
cov0 = 4
gam = 2
CovDiagPriorA3 = [cov0* (j**(-gam)) for j in list(range(1,TarDimExA3+1))]
CovPriorExA3 = Utils.mkDiagCov(CovDiagPriorA3)

#Information for Gaussian Posterior
CovPostExA3 =CovPriorExA3
pertdirA3= [1,1]
meanPostExA3 = np.concatenate([np.array(pertdirA3), np.zeros(TarDimExA3-len(pertdirA3))])

PostCovInvA3 = np.diag(1/np.array(CovDiagPriorA3))
PotExA3Pass = partial(Utils.PotGaussPertMean, Post_Mean = meanPostExA3, PostCovInv = PostCovInvA3,mode = "soft")

#runAtypeExperiment(expId, modDim, priorCov, postMean, postCov, PlotComps,burnIn=1000)

runAtypeExperiment(expIdA3,TarDimExA3, CovPriorExA3,meanPostExA3,CovPostExA3,PotExA3Pass,[0,1,4])


In [ ]:
#Set up Workflow for Type B experiments

def runBtypeExperiment(expId, fFnStr, modDim, z_data, noise_sig, priorCov, Pot, PlotComps, buildStatMeasure = True,burnIn = 1000):
    #Make Problem Information File
    ExpDir = "Large_p_study/Experiment_B/" + "Ex_ID_"+ str(expId) + "/"
    FileNmBase = saveLocBase+ExpDir
    FileNmBaseData =saveLocBaseData+ExpDir

    os.makedirs(FileNmBase, exist_ok=True)
    os.makedirs(FileNmBaseData, exist_ok=True)

    #Make Problem information String
    probinfoData  = "Target Type:  Model Problem B \n"
    probinfoData += "Model Dimension: " + str(modDim) + "\n"
    probinfoData += "Forward Map:  f(x,y) = " + fFnStr + "\n"
    probinfoData += "Data z: " + str(z_data) + "\n"
    probinfoData += "Observation Noise sigma: " + str(noise_sig)+ "\n"
    probinfoData += "Prior Covariance: " + "\n" + str(priorCov)

    Utils.message_ntfy("RUNNING EXAMPLE B" + str(expId) + "\n" + probinfoData)

    #Save Problem Information
    probDataFile = FileNmBase + "Problem_Info.txt"
    with open(probDataFile, 'w') as file:
        file.write(probinfoData)

    #Build or reload Stationary Measure
    statmeasureFnNm = FileNmBaseData + "stationary_seq.csv"
    BuildStatMeasure = True

    if BuildStatMeasure:
        print("Building Stationary Measure")
        Utils.message_ntfy("Building Stationary Measure")
        FinalSamps = make_stat_measure(modDim,priorCov,Pot)
        Utils.writeCSV(statmeasureFnNm, FinalSamps)
    
    else:
        print("Reloading Stationary Measure")
        Utils.message_ntfy("Reloading Stationary Measure")
        FinalSamps = Utils.readCSV(statmeasureFnNm)

    numStationarySamps= FinalSamps.shape[0]
    print("Number of samples available: " + str(numStationarySamps))

    histFileNm = FileNmBase + "Baseline_Histogram.png"
    R =5
    dr=.1
    Utils.makeHistGrid(R, dr, FinalSamps, modDim,histFileNm, C=priorCov, beta=0.95, hidePlt = True)

    q0_stat_draw = lambda nmICs: FinalSamps[rng.integers(0,numStationarySamps,size =nmICs)]

    run_p_Sweep_Batches(ImpLists, q0_stat_draw, modDim, priorCov, Pot, FileNmBase,FileNmBaseData,PlotComps,burnIn)

In [ ]:
#Experiment B1
expIdB1 = 1
#We consider a target of the type Model Problem B, [MM1] where
#    f(x,y) = y(x-a)^r 
NumParmsExB1 = 2
zExB1 = 6
aExB1 = .3
rExB1 = 1
sigExB1 = 1
CovExB1 = np.diag([3,2])

PotExB1Pass = partial(Utils.PotExB1, sig=sigExB1, a= aExB1, r=rExB1, z=zExB1,  mode="soft")
fFnStrB1 = "y (x - " + str(aExB1) + ")^" + str(rExB1) 

#runBtypeExperiment(expId, fFnStr, modDim, z_data, noise_sig, priorCov, Pot, PlotComps buildStatMeasure = True)

runBtypeExperiment(expIdB1,fFnStrB1,NumParmsExB1,zExB1,sigExB1,CovExB1,PotExB1Pass,[0,1])


In [ ]:
#Experiment B2
expIdB2 = 2 #Experiment ID

#We consider a target of the type Model Problem B, [MM1] where
#    f(x) = x* C^{-1} x
#The problem parameters are:
NumParmsExB2 = 2
cov0 = 4
gam = 2
CovDiag_p = [cov0* (j**(-gam)) for j in list(range(1,NumParmsExB2+1))]
CovInvDiag_p = [cov0**(-1) * (j**(gam)) for j in list(range(1,NumParmsExB2+1))]
CovPriorExB2 = Utils.mkDiagCov(CovDiag_p)
CovPriorInvExB2 = Utils.mkDiagCov(CovInvDiag_p)

sigExB2 = .5

#Compute zExB2 so that \mu_0({x | x^* C^{-1} x \leq zBxB2\}) = prob_below
prob_below = .95
zExB2 = chi2.ppf(prob_below, df=NumParmsExB2)  


#Make Loglikehood function
compDimB2 =2
PotExB2Pass = partial(Utils.PotMahalanobis, compDim = compDimB2, sig=sigExB2, CovInv = CovPriorInvExB2,  zdata = zExB2, mode="soft")
fFnStrB2 = "x* C^{-1} x"

#runBtypeExperiment(expId, fFnStr, modDim, z_data, noise_sig, priorCov, Pot, PlotComps buildStatMeasure = True)
runBtypeExperiment(expIdB2, fFnStrB2, NumParmsExB2, zExB2, sigExB2, CovPriorExB2,PotExB2Pass,[0,1])


In [ ]:
#Experiment B3
expIdB3 = 3 #Experiment ID

#We consider a target of the type Model Problem B, where
#    f(x) = \pi(x)* (pi C)^{-1} \pi(x)
#where \pi(x_1,...,x_N) = (x_1,x_2) is the projection onto the first two dimensions

NumParmsExB3 = 7
cov0 = 4
gam = 2
CovPriorExB3 = Utils.mkDiagCov([cov0* (j**(-gam)) for j in list(range(1,NumParmsExB3+1))])
compDimB3 =2
CovPriorInvTrucB3 =Utils.mkDiagCov([cov0**(-1) * (j**(gam)) for j in list(range(1,compDimB3+1))])

#Compute zExB3 so that \mu_0({\pi(x) | \pi(x)^* C^{-1} \pi(x) \leq zBxB2\}) = prob_below
informedDirB3 = 2
prob_below = .95
zExB3 = chi2.ppf(prob_below, df=informedDirB3)  

sigExB3 = .5

PotExB3Pass = partial(Utils.PotMahalanobis, compDim = compDimB3, sig=sigExB3, CovInv = CovPriorInvTrucB3,  zdata = zExB3, mode="soft")
fFnStrB3 = "P_2(x)* C^{-1} P_2x where P_2(x) =(x_1,x_2)"

runBtypeExperiment(expIdB3, fFnStrB3, NumParmsExB3, zExB3, sigExB3, CovPriorExB3, PotExB3Pass,[0,1,2,3,4])

In [2]:
%load_ext autoreload
%autoreload 2

#Set up Workflow for Type C experiments

    
#Set-up: Packages to Import and Untility Functions

#Core Numerical Packages for Paral
import MCMC_Sampliers_Testing as MCMCsmp
import Utilities as Utils


def runCtypeExperiment(expId, numParams, g_force, kappa, z_data, noise_sig, priorCov, Pot, PlotComps, buildStatMeasure = True,burnIn = 1000, runParameterSweap = True):
    #Make Problem Information File
    ExpDir = "Large_p_study/Experiment_C/" + "Ex_ID_"+ str(expId) + "/"
    FileNmBase = saveLocBase+ExpDir
    FileNmBaseData =saveLocBaseData+ExpDir

    os.makedirs(FileNmBase, exist_ok=True)
    os.makedirs(FileNmBaseData, exist_ok=True)

    #Make Problem information String
    probinfoData  = "Target Type:  Model Problem C \n"
    probinfoData += "Model Dimension: " + str(numParams) + "\n"
    probinfoData += "g: " + str(g_force) + "\n"
    probinfoData += "kappa: " + str(kappa)+ "\n"    
    probinfoData += "Data z: " + str(z_data) + "\n"
    probinfoData += "Observation Noise sigma: " + str(noise_sig)+ "\n"
    probinfoData += "Prior Covariance: " + "\n" + str(priorCov)



    Utils.message_ntfy("RUNNING EXAMPLE C" + str(expId) + "\n" + probinfoData)

    #Save Problem Information
    probDataFile = FileNmBase + "Problem_Info.txt"
    with open(probDataFile, 'w') as file:
        file.write(probinfoData)

    #Build or reload Stationary Measure
    statmeasureFnNm = FileNmBaseData + "stationary_seq.csv"
    BuildStatMeasure = True

    if BuildStatMeasure:
        print("Building Stationary Measure")
        Utils.message_ntfy("Building Stationary Measure")
        FinalSamps = make_stat_measure(numParams,priorCov,Pot)
        #FinalSamps = make_stat_measure(numParams,priorCov,Pot, numChain =5, chainLn =2000)
        Utils.writeCSV(statmeasureFnNm, FinalSamps)
    
    else:
        print("Reloading Stationary Measure")
        Utils.message_ntfy("Reloading Stationary Measure")
        FinalSamps = Utils.readCSV(statmeasureFnNm)

    numStationarySamps= FinalSamps.shape[0]
    print("Number of samples available: " + str(numStationarySamps))

    histFileNm = FileNmBase + "Baseline_Histogram.png"
    R =5
    dr=.1
    Utils.makeHistGrid(R, dr, FinalSamps, numParams,histFileNm, C=priorCov, beta=0.95, hidePlt = True)

    Utils.generate_Random_Rot_Hist(numParams, FinalSamps,FileNmBase, R = 5, dr = .1, numRots=10)

    if runParameterSweap:

        q0_stat_draw = lambda nmICs: FinalSamps[rng.integers(0,numStationarySamps,size =nmICs)]
        
        run_p_Sweep_Batches(ImpLists, q0_stat_draw, numParams, priorCov, Pot, FileNmBase,FileNmBaseData,PlotComps,burnIn)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
#Experiment C1: Set-up
expIdC1 = 1 #Experiment ID

#We use precisely the example appearing in refer [1]
#We Specifying Problem Parameters as follows
#Model dimension and parameter space size are
ModDmExC1= 4
NumParmsExC1 = int(ModDmExC1*(ModDmExC1 -1)/2)

 
gvecExC1 = np.array([.1,0,5,2])
sigExC1 = 2
zExC1 = np.array([4.601,18.021,0,0])
dataDimExC1 = 2
kapExC1 = .05

# Covariance of the 'prior' C = cov0[1^{-gam}, 2^[-gam],..., N^{-gam}]
cov0 = 5
gam = 1.5
CovExC1 = np.diag([cov0* (j**(-gam)) for j in list(range(1,NumParmsExC1+1))])

PotExC1Pass = partial(Utils.PotExAD, gvec=gvecExC1, sig=sigExC1, ModDm=ModDmExC1, z=zExC1, kap=kapExC1, dataDim=dataDimExC1, mode = "soft")

runCtypeExperiment(expIdC1, NumParmsExC1, gvecExC1, kapExC1, zExC1, sigExC1, CovExC1, PotExC1Pass, [0,1,2,3,4,5], buildStatMeasure = True,burnIn = 1000)


In [ ]:
#Experiment C2: Set-up
expIdC2 = 2 #Experiment ID

#Model dimension and parameter space size are
ThetaDmExC2= 5
NumParmsC2 = int(ThetaDmExC2*(ThetaDmExC2 -1)/2)

kapExC2 = 0.01 #Make kap smaller to get a stiffer problem
sigExC2 = 0.01
cov0 = 5
gam = 1.1
dataDimExC2 = 2
CovPriorC2 = np.diag([cov0* (j**(-gam)) for j in list(range(1,NumParmsC2+1))])


ExpDir = "Large_p_study/Experiment_C/" + "Ex_ID_"+ str(expIdC2) + "/"
FileNmBase = saveLocBase+ExpDir
FileNmBaseData =saveLocBaseData+ExpDir

os.makedirs(FileNmBase, exist_ok=True)
os.makedirs(FileNmBaseData, exist_ok=True)

aMatchDictfileLoc = FileNmBaseData + "A_match_dict.npy"

fnd_match = True
if fnd_match:
    gMean= np.array([1,0,0,0,0])
    gCov = np.diag([1,.1,.1,.1,.1])
    tol = .005
    #tol = .1
    lamC2 = .9
    transFn = partial(Utils.rot_A, numParm = NumParmsC2, lam = lamC2)
    obsComp = np.array([1,1,0,0,0])
    ObsOp = partial(Utils.getComps, compArray=obsComp)
    matchbool, match_dict, total_tries = Utils.Find_AD_Match_Parallel(NumParmsC2, ThetaDmExC2, gMean, gCov, CovPriorC2, kapExC2, tol, transFn, ObsOp)

    print("total tries:", total_tries)
    
    np.save(aMatchDictfileLoc, match_dict, allow_pickle=True)  #Saving dictionary

    gC2 = match_dict["g"]
    A_tru = match_dict["A"]
    #print(A_tru)
    zC2 = Utils.getThA(ThetaDmExC2,A_tru,gC2,kapExC2)*obsComp

else:
    match_dict = np.load(aMatchDictfileLoc, allow_pickle=True).item()

    gC2 = match_dict["g"]
    A_tru = match_dict["A"]
    zC2 = Utils.getThA(A_tru,gC2,kapExC2)


gvecExC2 = gC2
zExC2 = zC2
dataDimExC2 = 2


PotExC2Pass = partial(Utils.PotExAD, gvec=gvecExC2, sig=sigExC2, ModDm=ThetaDmExC2, z=zExC2, kap=kapExC2, dataDim=dataDimExC2, mode = "soft")

runCtypeExperiment(expIdC2, NumParmsC2, gvecExC2, kapExC2, zExC2[0:dataDimExC2], sigExC2, CovPriorC2, PotExC2Pass, list(range(NumParmsC2)), buildStatMeasure = True,burnIn = 1000,runParameterSweap=False)



round 1/48 | total iters ~0 | best(worker) inf | best(global) inf

In [ ]:
#Experiment C2: Set-up
expIdC2 = 2 #Experiment ID

#Model dimension and parameter space size are
ThetaDmExC2= 51
NumParmsC2 = int(ThetaDmExC2*(ThetaDmExC2 -1)/2)